In [10]:
import json
import os
import shutil
from kafka.admin import KafkaAdminClient, NewTopic

def run_cleanup(config_file='config.json'):
    # 1. Load your configuration
    with open(config_file, 'r') as file:
        config = json.load(file)

    bootstrap_servers = config.get("kafka_bootstrap_servers", "localhost:9092")
    topics_to_delete = [
        config.get("kitchen_station_events", "kitchen_station_events"),
        config.get("dispatch_events", "dispatch_events")
    ]
    
    # Get the local paths from config
    local_output = "./output" # Main output folder
    
    print("--- Starting Project Cleanup ---")

    # 3. Delete Kafka Topics
    admin_client = None
    try:
        admin_client = KafkaAdminClient(bootstrap_servers=bootstrap_servers)
        # DELETE existing topics
        print(f"Deleting topics: {topics_to_delete}...")
        admin_client.delete_topics(topics=topics_to_delete)
        
        # Pause for a moment to let Kafka process the deletion
        import time
        time.sleep(4) 
        
        # CREATE fresh topics
        # num_partitions=1 and replication_factor=1 are standard for local development
        new_topics = [
            NewTopic(name=config.get("kitchen_station_events", "kitchen_station_events"), num_partitions=1, replication_factor=1),
            NewTopic(name=config.get("dispatch_events", "dispatch_events"), num_partitions=1, replication_factor=1)
        ]
        
        admin_client.create_topics(new_topics=new_topics)
        print(f"Successfully recreated topics: {topics_to_delete}")
        
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if admin_client:
            admin_client.close() # Manually close the connection
            print("Admin client connection closed safely.")

    # 4. Delete HDFS Data (For WSL HDFS)
    hdfs_output_path = "./output" 

    print(f"Attempting to clean HDFS path: {hdfs_output_path}")
    
    # This command tells WSL to run the HDFS delete command
    hdfs_cmd = f"hdfs dfs -rm -r -skipTrash {hdfs_output_path}/*"
    
    response = os.system(hdfs_cmd)
    
    if response == 0:
        print("Successfully cleaned HDFS.")
    else:
        print("Note: HDFS path was already empty or HDFS is not running.")

    print("--- Cleanup Finished. You are ready for a new run! ---")

if __name__ == "__main__":
    run_cleanup()

--- Starting Project Cleanup ---
Deleting topics: ['kitchen_station_events', 'dispatch_events']...
Successfully recreated topics: ['kitchen_station_events', 'dispatch_events']
Admin client connection closed safely.
Attempting to clean HDFS path: ./output
Deleted output/checkpoints
Deleted output/dispatch_data
Deleted output/kitchen_data
Successfully cleaned HDFS.
--- Cleanup Finished. You are ready for a new run! ---
